In [17]:
import numpy as np
import pandas as pd
import snowflake.connector
import requests
import io

In [20]:
conn = snowflake.connector.connect(user='esteban.correa@RAPPI.COM', 
                                   authenticator='externalbrowser', 
                                   account='hg51401', 
                                   warehouse="RP_PERSONALUSER_WH",
                                   database="FIVETRAN")

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://fs.security.rappi.com/adfs/ls?SAMLRequest=jZJRb5swFIX%2FCvKeg01gU2IlqWijNJmyjTbpNO3NA0OtGJv52tDs189AI7UPrfpm2d%2FxPfeeu7h6qmXQcgNCqyWKQoICrnJdCFUt0cNxM5mhACxTBZNa8SU6c0BXqwWwWjY0dfZR3fO%2FjoMN%2FEcKaP%2BwRM4oqhkIoIrVHKjN6SH9tqfTkFAGwI315dALSfO%2BpjHa6lzLi6QA4e09WttQjLuuC7s41KbCU0IIJnPsqR75dOGffE9v8BEmSc97wuPZc6FrocYRvOfqzwgB3R6P2ST7cTiiIL10d6MVuJqbAzetyPnD%2FX40AN7B9vZzlJAoBKW7UrITz3XdOOv%2FCv0Jl7zAUlfCT2i3XqLmJIp592uT3JI2dlV9%2FXU7s%2FJflrLqdNfu27VrmRYGosTdxdksR8HPS57TPs8dgOM71ado%2FRWZJhPyZULiYzSn8ZQmHpqT3yhY%2BxSFYnZQXqyWEALPnRH2HBrWNGJwyIoSsAQ0LgIdKpjVx9pb4Jea50367oe7W2daivwcbLSpmX179lEYDTeimJQDSnnNhEyLwnAAn4GUursxnFm%2FsNY4jvBqrPp6ZVf%2FAQ%3D%3D&RelayState=51075 to authenticate...


# Inventario Redash COMPLETO NO FRUVER

In [23]:
url='http://redash.rappi.com/api/queries/87502/results.csv?api_key=XVM2nMR8n7jP04S8UJUMapMT7CkxNJjvSvnQLDSe'
urlData = requests.get(url).content
inventario = pd.read_csv(io.StringIO(urlData.decode('utf-8')))


C:\Users\juane\AppData\Local\Temp\ipykernel_17308\148215383.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  inventario = pd.read_csv(io.StringIO(urlData.decode('utf-8')))


In [24]:
print(inventario.columns)

Index(['cruce', 'sync_wh', 'warehouse_id', 'warehouse_name',
       'storereferenceid', 'sync_id', 'product_name', 'ean', 'stock'],
      dtype='object')


In [25]:
inventario.drop(['cruce','warehouse_name','ean'],axis=1,inplace=True)

In [26]:
inventario.rename(columns={
    'sync_wh': 'SYNC_WH',
    'warehouse_id': 'WAREHOUSEID',
    'storereferenceid': 'STOREREFERENCEID',
    'sync_id': 'SYNC_ID',
    'product_name': 'PRODUCT_NAME',
    'stock': 'INVENTARIO_TURBO'
}, inplace=True)


# SWA - AVL

In [28]:
SWA = """with 

avl_closing as (
select final.warehouseid as warehouse_id, final.sync_wh_id, d.external_id as dependencia, final.warehouse_name, final.storereferenceid as storereference_id, final.sync_product_id,
e.ean, final.sales_ayer, final.sales_28, final.full_sales_28, final.available_hours, final.total_hours, final.full_available_hours, final.full_total_hours
from 
(select a.warehouseid, w.warehouse_id as sync_wh_id, a.warehousename as warehouse_name, a.storereferenceid, p.product_id as sync_product_id, a.sales_units as sales_ayer, a.sales_28, a.available_hours, a.total_hours, a.full_available_hours, a.full_total_hours, a.full_sales_28, a.wl_type
from fivetran.cpgs_turbo_ds_public.global_closing_inventory_current as a
left join fivetran.co_amysql_turbo_emergency_order_turbo_sync.warehouse_integration as w
on w.external_id = a.warehouseid
left join fivetran.co_amysql_turbo_emergency_order_turbo_sync.product_integration as p
on p.external_id = a.storereferenceid
where a.country_code = 'CO' 
and a.main_date = current_date - interval '1 day'
and w.integration_name = 'vivo'
and p.integration_name = 'vivo') as final
left join fivetran.co_amysql_turbo_emergency_order_turbo_sync.warehouse_integration as d
on d.warehouse_id = final.sync_wh_id
left join fivetran.co_amysql_turbo_emergency_order_turbo_sync.product as e
on e.id = final.sync_product_id
where d.integration_name = 'exito'),

closing_2 as (
select a.*, b.vertical, b.atribute, b.product_name, b.macrocategory_name, b.category_name, b.subcategory_name, b.trademark_name, b.supplier_name, b.maker_name, b.bucket
from avl_closing as a
left join fivetran.cpgs_turbo_ds_public.global_product_supply as b
on a.warehouse_id = b.warehouseid and a.storereference_id = b.storereferenceid
having b.wl_type in ('1 Ideal','3 Substitute') 
and b.active = 'Active' 
--and is_cedi = 'False' 
--and atribute = 'No R2E' 
--and product_name != 'Out of catalogo' 
and b.vertical = 'Turbo'
and b.country_code = 'CO'
and b.supplier_name NOT LIKE 'No Supplier%'
--and b.supplier_name NOT LIKE 'Almacenes Ex%'
and b.category_name LIKE 'Fruta%'),

wl as (
select city, warehouseid, storereferenceid, scope, wl_type
from fivetran.cpgs_turbo_ds_public.global_wishlist_with_pareto_new
where country_code = 'CO'),

closing_3 as (
select *, case when full_sales_28 < 0 then 0 else full_sales_28 end as vt, w.city as city_name, w.scope as scope_wh, w.wl_type as is_wl
from closing_2 as c
left join wl as w
on w.warehouseid = c.warehouse_id and w.storereferenceid = c.storereference_id),

full_sales_28 as (
select sum(vt) as total_sales
from closing_3),

recepcion as (
select cast(last_value(k.created_at) over (partition by l.warehouse_id, l.product_id order by k.created_at) as date) as reception_date,
l.warehouse_id, l.product_id, k.lot_stock
from fivetran.co_amysql_turbo_emergency_order_turbo_inventory_savvy_ms.kardex as k
left join fivetran.co_amysql_turbo_emergency_order_turbo_inventory_savvy_ms.lot as l 
on k.lot_id = l.id
left join fivetran.co_amysql_turbo_emergency_order_turbo_inventory_savvy_ms.kardex_type as kt
on kt.id = k.kardex_type_id 
where k.kardex_type_id in (1,19)),

transformation as (
select transformation_date, warehouse_id, product_id, sum(transfor_units) as transfor_units
from (
select last_value(k.created_at) over (partition by l.warehouse_id, l.product_id order by k.created_at) as transformation_date,
l.warehouse_id, l.product_id, k.lot_stock as transfor_units
from fivetran.co_amysql_turbo_emergency_order_turbo_inventory_savvy_ms.kardex as k
left join fivetran.co_amysql_turbo_emergency_order_turbo_inventory_savvy_ms.lot as l 
on k.lot_id = l.id
left join fivetran.co_amysql_turbo_emergency_order_turbo_inventory_savvy_ms.kardex_type as kt
on kt.id = k.kardex_type_id 
where k.kardex_type_id in (5) )
group by transformation_date, warehouse_id, product_id)

select c.city_name, c.warehouse_id, c.sync_wh_id, c.dependencia, c.warehouse_name, c.storereference_id, c.sync_product_id, c.ean, c.is_wl, c.scope_wh, c.product_name, 
c.macrocategory_name, c.category_name, c.subcategory_name, c.supplier_name, c.trademark_name, c.maker_name, c.bucket, c.sales_ayer, c.sales_28, 
round(div0(sum(c.full_available_hours),sum(c.full_total_hours))*100,2) as avl_nacional, 
to_varchar(zeroifnull(round((c.vt * (c.full_available_hours / nullif(c.full_total_hours,0)) / nullif((select * from full_sales_28), 0)) * 100, 6)), '0.00000000') as swa_nacional,
(to_varchar(zeroifnull(round((c.vt / nullif((select * from full_sales_28), 0)) * 100, 6)), '0.00000000') - swa_nacional) as swa_missing, r.reception_date, t.transformation_date, 
t.transfor_units
from closing_3 as c
left join recepcion as r
on r.warehouse_id = c.sync_wh_id and r.product_id = c.sync_product_id
left join transformation as t
on t.warehouse_id = c.sync_wh_id and t.product_id = c.sync_product_id
group by c.city_name, c.warehouse_id, c.sync_wh_id, c.dependencia, c.warehouse_name, c.storereference_id, c.sync_product_id, c.ean, c.is_wl, c.scope_wh, c.product_name,
c.macrocategory_name, c.category_name, c.subcategory_name, c.supplier_name, c.trademark_name, c.maker_name, c.bucket, c.sales_ayer, c.sales_28, c.vt, c.full_available_hours, c.full_total_hours,
r.reception_date, t.transformation_date, t.transfor_units
order by swa_missing desc;
"""

In [8]:
df_swa = pd.read_sql(SWA, conn)
df_swa.head(2)

C:\Users\marcel.kempe\AppData\Local\Temp\ipykernel_29792\4247655679.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_swa = pd.read_sql(SWA, conn)


,CITY_NAME,WAREHOUSE_ID,SYNC_WH_ID,DEPENDENCIA,WAREHOUSE_NAME,STOREREFERENCE_ID,SYNC_PRODUCT_ID,EAN,IS_WL,SCOPE_WH,...,MAKER_NAME,BUCKET,SALES_AYER,SALES_28,AVL_NACIONAL,SWA_NACIONAL,SWA_MISSING,RECEPTION_DATE,TRANSFORMATION_DATE,TRANSFOR_UNITS
0,BogotC!Exito,180,12,4841,El Lago,431098,3396,2200000313515,1 Ideal,Bogotá Diamante,...,Out of catalogo,S,110.0,6620.0,25.0,0.27038900,0.811165,2024-04-01,NaT,NaN
1,Medellín,236,52,4854,Tesoro,431087,3747,2200000313409,1 Ideal,MED TESORO,...,Out of catalogo,S,0.0,4432.0,0.0,0.00000000,0.724086,2024-03-31,2024-03-31 11:08:10+00:00,711.0


In [9]:
len(df_swa)

4370

# NIVELES DE SERVICIO

In [10]:
ns = """
  WITH ns_lw AS (
    SELECT 
        WAREHOUSE_ID,
        PRODUCT_ID AS STOREREFERENCEID,
        SUPPLIER_ID AS PROVIDERID,
        SUM(pod.quantity) AS planned_lw,
        SUM(pod.received) AS delivered_lw
    FROM 
        fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER PO
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_DETAIL POD ON PO.ID = POD.PURCHASE_ORDER_ID
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_STATUS POS ON POS.ID = PO.STATUS_ID
    WHERE 
        DATE(expected_at) BETWEEN CURRENT_DATE - INTERVAL '8 DAY' AND CURRENT_DATE - INTERVAL '1 DAY'
    GROUP BY
        WAREHOUSE_ID,
        PRODUCT_ID,
        SUPPLIER_ID
),
ns_1 AS (
    SELECT 
        WAREHOUSE_ID,
        PRODUCT_ID AS STOREREFERENCEID,
        SUPPLIER_ID AS PROVIDERID,
        SUM(pod.quantity) AS planned_ayer,
        SUM(pod.received) AS delivered_ayer
    FROM 
        fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER PO
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_DETAIL POD ON PO.ID = POD.PURCHASE_ORDER_ID
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_STATUS POS ON POS.ID = PO.STATUS_ID
    WHERE 
        DATE(expected_at) = CURRENT_DATE - INTERVAL '1 DAY'
    GROUP BY
        WAREHOUSE_ID,
        PRODUCT_ID,
        SUPPLIER_ID
),
ns_2 AS (
    SELECT 
        WAREHOUSE_ID,
        PRODUCT_ID AS STOREREFERENCEID,
        SUPPLIER_ID AS PROVIDERID,
        SUM(pod.quantity) AS planned_2,
        SUM(pod.received) AS delivered_2
    FROM 
        fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER PO
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_DETAIL POD ON PO.ID = POD.PURCHASE_ORDER_ID
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_STATUS POS ON POS.ID = PO.STATUS_ID
    WHERE 
        DATE(expected_at) = CURRENT_DATE - INTERVAL '2 DAY'
    GROUP BY
        WAREHOUSE_ID,
        PRODUCT_ID,
        SUPPLIER_ID
),
ns_3 AS (
    SELECT 
        WAREHOUSE_ID,
        PRODUCT_ID AS STOREREFERENCEID,
        SUPPLIER_ID AS PROVIDERID,
        SUM(pod.quantity) AS planned_3,
        SUM(pod.received) AS delivered_3
    FROM 
        fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER PO
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_DETAIL POD ON PO.ID = POD.PURCHASE_ORDER_ID
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_STATUS POS ON POS.ID = PO.STATUS_ID
    WHERE 
        DATE(expected_at) = CURRENT_DATE - INTERVAL '3 DAY'
    GROUP BY
        WAREHOUSE_ID,
        PRODUCT_ID,
        SUPPLIER_ID
),
ns_4 AS (
    SELECT 
        WAREHOUSE_ID,
        PRODUCT_ID AS STOREREFERENCEID,
        SUPPLIER_ID AS PROVIDERID,
        SUM(pod.quantity) AS planned_4,
        SUM(pod.received) AS delivered_4
    FROM 
        fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER PO
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_DETAIL POD ON PO.ID = POD.PURCHASE_ORDER_ID
        LEFT JOIN fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_STATUS POS ON POS.ID = PO.STATUS_ID
    WHERE 
        DATE(expected_at) = CURRENT_DATE - INTERVAL '4 DAY'
    GROUP BY
        WAREHOUSE_ID,
        PRODUCT_ID,
        SUPPLIER_ID
)
SELECT 
    lw.warehouse_id,
    lw.storereferenceid,
    lw.providerid,
    lw.planned_lw,
    lw.delivered_lw,
    ns1.planned_ayer,
    ns1.delivered_ayer,
    ns2.planned_2,
    ns2.delivered_2,
    ns3.planned_3,
    ns3.delivered_3,
    ns4.planned_4,
    ns4.delivered_4
FROM
    ns_lw AS lw
    LEFT JOIN
    ns_1 AS ns1 on ns1.warehouse_id = lw.warehouse_id and ns1.storereferenceid = lw.storereferenceid
    LEFT JOIN
    ns_2 AS ns2 on ns2.warehouse_id = lw.warehouse_id and ns2.storereferenceid = lw.storereferenceid
    LEFT JOIN
    ns_3 AS ns3 on ns3.warehouse_id = lw.warehouse_id and ns3.storereferenceid = lw.storereferenceid
    LEFT JOIN
    ns_4 AS ns4 on ns4.warehouse_id = lw.warehouse_id and ns4.storereferenceid = lw.storereferenceid 
;
"""

df_ns1 = pd.read_sql(ns, conn)
df_ns1.head()

C:\Users\marcel.kempe\AppData\Local\Temp\ipykernel_29792\1434457293.py:119: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ns1 = pd.read_sql(ns, conn)


,WAREHOUSE_ID,STOREREFERENCEID,PROVIDERID,PLANNED_LW,DELIVERED_LW,PLANNED_AYER,DELIVERED_AYER,PLANNED_2,DELIVERED_2,PLANNED_3,DELIVERED_3,PLANNED_4,DELIVERED_4
0,93,638,212,8,2,NaN,NaN,NaN,NaN,NaN,NaN,4.0,0.0
1,93,1925,212,12,3,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0.0
2,93,9098,212,33,11,NaN,NaN,NaN,NaN,NaN,NaN,22.0,11.0
3,93,2064,212,4,2,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2.0
4,93,1918,212,4,2,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2.0


In [11]:
len(df_ns1)

15870

# FORECAST PASADO + VENTAS

In [12]:
forecast = """with forecast_lw as (
select vivo_warehouse_id as warehouseid, vivo_product_id as storereferenceid, sum(forecast) as forecast_lw, sum(sales_units) as sales_lw
from fivetran.cpgs_turbo_ds_public.global_forecast_main
where country = 'CO'
AND date BETWEEN current_date - INTERVAL '8 day' AND current_date - INTERVAL '2 day'
group by vivo_warehouse_id,
  vivo_product_id),

forecast_ayer as (
select vivo_warehouse_id as warehouseid, vivo_product_id as storereferenceid, sum(forecast) as forecast_ayer, sum(sales_units) as sales_ayer
from fivetran.cpgs_turbo_ds_public.global_forecast_main
where country = 'CO'
 AND date = current_date() - interval '1 day'
group by warehouseid, storereferenceid),

forecast_2 as (
select vivo_warehouse_id as warehouseid, vivo_product_id as storereferenceid, sum(forecast) as forecast_menos2, sum(sales_units) as sales_menos2
from fivetran.cpgs_turbo_ds_public.global_forecast_main
where country = 'CO'
 AND date = current_date() - interval '2 day'
group by warehouseid, storereferenceid),

forecast_3 as (
select vivo_warehouse_id as warehouseid, vivo_product_id as storereferenceid, sum(forecast) as forecast_menos3, sum(sales_units) as sales_menos3
from fivetran.cpgs_turbo_ds_public.global_forecast_main
where country = 'CO'
 AND date = current_date() - interval '3 day'
group by warehouseid, storereferenceid)

SELECT 
f.warehouseid,
f.storereferenceid,
f.forecast_lw,
f.sales_lw,
f1.forecast_ayer,
f1.sales_ayer,
f2.forecast_menos2,
f2.sales_menos2,
f3.forecast_menos3,
f3.sales_menos3
FROM
    forecast_lw AS f
    LEFT JOIN
    forecast_ayer AS f1 on f1.warehouseid = f.warehouseid and f1.storereferenceid = f.storereferenceid
    LEFT JOIN
    forecast_2 AS f2 on f2.warehouseid = f.warehouseid and f2.storereferenceid = f.storereferenceid
    LEFT JOIN
    forecast_3 AS f3 on f3.warehouseid = f.warehouseid and f3.storereferenceid = f.storereferenceid;

"""
df_forecast = pd.read_sql(forecast, conn)
df_forecast.head()

C:\Users\marcel.kempe\AppData\Local\Temp\ipykernel_29792\681383390.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_forecast = pd.read_sql(forecast, conn)


,WAREHOUSEID,STOREREFERENCEID,FORECAST_LW,SALES_LW,FORECAST_AYER,SALES_AYER,FORECAST_MENOS2,SALES_MENOS2,FORECAST_MENOS3,SALES_MENOS3
0,296,432315,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,231,442524,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,286,427521,3.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
3,180,434249,23.0,4.0,3.0,0.0,3.0,0.0,3.0,0.0
4,246,439077,11.0,8.0,1.0,0.0,2.0,3.0,1.0,1.0


# MERMA

In [13]:
merma_ayer = """SELECT 
    a.WAREHOUSEID,
    a.STOREREFERENCEID,
    a.UNITS as Merma_LW,
    b.UNITS as Merma_Ayer,
    c.UNITS as Merma_2,
    d.UNITS as Merma_3
    
FROM fivetran.cpgs_turbo_ds_public.global_know_shrinkage a
LEFT JOIN
fivetran.cpgs_turbo_ds_public.global_know_shrinkage b
ON
a.WAREHOUSEID = b.WAREHOUSEID
and a.STOREREFERENCEID = b.STOREREFERENCEID
left join 
fivetran.cpgs_turbo_ds_public.global_know_shrinkage c
ON
a.WAREHOUSEID = c.WAREHOUSEID
and a.STOREREFERENCEID = c.STOREREFERENCEID
left join 
fivetran.cpgs_turbo_ds_public.global_know_shrinkage d
ON
a.WAREHOUSEID = c.WAREHOUSEID
and a.STOREREFERENCEID = c.STOREREFERENCEID
WHERE 
    a.COUNTRY_CODE IN ('CO')
    AND b.COUNTRY_CODE IN ('CO')
    AND c.COUNTRY_CODE IN ('CO')
    AND d.COUNTRY_CODE IN ('CO')
    AND a.MAIN_DATE BETWEEN CURRENT_DATE - INTERVAL '8 DAY' AND CURRENT_DATE
    AND b.MAIN_DATE = DATEADD(DAY, -1, CURRENT_DATE())
    AND c.MAIN_DATE = DATEADD(DAY, -2, CURRENT_DATE())
    AND d.MAIN_DATE = DATEADD(DAY, -3, CURRENT_DATE());
    """
df_merma_ayer = pd.read_sql(merma_ayer, conn)
df_merma_ayer.head()

C:\Users\marcel.kempe\AppData\Local\Temp\ipykernel_29792\249409485.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_merma_ayer = pd.read_sql(merma_ayer, conn)


,WAREHOUSEID,STOREREFERENCEID,MERMA_LW,MERMA_AYER,MERMA_2,MERMA_3


# ROQ

In [14]:
ROQ = """select *
from fivetran.turbo_turbo_supply.roq_direct_supply_sync 
where country = 'CO' 
and date = current_date;
"""
df_roq = pd.read_sql(ROQ, conn)
df_roq.head()
#AND IS_PO_DAY = 'true'

C:\Users\marcel.kempe\AppData\Local\Temp\ipykernel_29792\3989557996.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_roq = pd.read_sql(ROQ, conn)


,CREATED_AT,ROQ_STARTED_AT,DATE,COUNTRY,CITY,WAREHOUSE,WAREHOUSE_ID,PROVIDER_ID,PROVIDER_NAME,RETAIL_ID,...,MOV_CHECK,ROQ_FINAL_BOXES,MOP,MOP_CHECK,AVAILABLE_TO_ORDER,MIN_ORDER_FILL,FILL_METHOD,FILLED_PERCENTAGE,ROQ_FINAL_UNITS_FILLED,ROQ_FINAL_VALUE_FILLED
0,2024-04-02 14:33:40.739027,2024-04-02 13:47:16.538978,2024-04-02,CO,BogotC!Exito,Chapinero,47,51,Almacenes Exito S.A.,8745,...,False,0,1,False,False,False,,0.0,0,0.0
1,2024-04-02 14:33:40.739027,2024-04-02 13:47:16.538978,2024-04-02,CO,BogotC!Exito,Chapinero,47,51,Almacenes Exito S.A.,8748,...,False,0,1,False,False,False,,0.0,0,0.0
2,2024-04-02 14:33:40.739027,2024-04-02 13:47:16.538978,2024-04-02,CO,BogotC!Exito,Chapinero,47,51,Almacenes Exito S.A.,8750,...,False,0,1,False,False,False,,0.0,0,0.0
3,2024-04-02 14:33:40.739027,2024-04-02 13:47:16.538978,2024-04-02,CO,BogotC!Exito,Chapinero,47,51,Almacenes Exito S.A.,8751,...,False,0,1,False,False,False,,0.0,0,0.0
4,2024-04-02 14:33:40.739027,2024-04-02 13:47:16.538978,2024-04-02,CO,BogotC!Exito,Chapinero,47,51,Almacenes Exito S.A.,8754,...,False,0,1,False,False,False,,0.0,0,0.0


# Ventas

In [15]:
venta = """WITH ventaslw AS (
    SELECT
        cast(created_at as date) as date,
        warehouse_id,
        retail_id,
        SUM(quantity) AS ventalw
    FROM
        fivetran.turbo_turbo_supply.co_product_demand_v1
    WHERE
        date BETWEEN current_date() - 8 AND current_date() - 1
    GROUP BY
        warehouse_id,
        retail_id,
        date
),
ventas_ayer AS (
    SELECT
        cast(created_at as date) as date,
        warehouse_id,
        retail_id,
        SUM(quantity) AS venta_ayer
    FROM
        fivetran.turbo_turbo_supply.co_product_demand_v1
    WHERE
        date = current_date() - 1
    GROUP BY
        warehouse_id,
        retail_id,
        date
),
ventas_2 AS (
    SELECT
        cast(created_at as date) as date,
        warehouse_id,
        retail_id,
        SUM(quantity) AS venta_2
    FROM
        fivetran.turbo_turbo_supply.co_product_demand_v1
    WHERE
        date = current_date() - 2
    GROUP BY
        warehouse_id,
        retail_id,
        date
),
ventas_3 AS (
    SELECT
        cast(created_at as date) as date,
        warehouse_id,
        retail_id,
        SUM(quantity) AS venta_3
    FROM
        fivetran.turbo_turbo_supply.co_product_demand_v1
    WHERE
        date = current_date() - 3
    GROUP BY
        warehouse_id,
        retail_id,
        date
),
product_rappi AS (
    SELECT
        product_id AS sync_product,
        external_id AS storereferenceid,
        integration_name
    FROM
        fivetran.co_amysql_turbo_emergency_order_turbo_sync.product_integration
    WHERE
        integration_name IN ('vivo')
),
wh_rappi AS (
    SELECT
        warehouse_id AS sync_wh,
        external_id AS warehouseid
    FROM
        fivetran.co_amysql_turbo_emergency_order_turbo_sync.warehouse_integration
    WHERE
        integration_name IN ('vivo')
)

SELECT
    v.warehouse_id AS sync_wh,
    w.warehouseid,
    v.retail_id AS sync_product,
    p.storereferenceid,
    v.ventalw,
    v1.venta_ayer,
    v2.venta_2,
    v3.venta_3
FROM
    ventaslw AS v
LEFT JOIN
    ventas_ayer AS v1 ON v1.warehouse_id = v.warehouse_id AND v1.retail_id = v.retail_id AND v1.date = v.date
LEFT JOIN
    ventas_2 AS v2 ON v2.warehouse_id = v.warehouse_id AND v2.retail_id = v.retail_id AND v2.date = v.date
LEFT JOIN
    ventas_3 AS v3 ON v3.warehouse_id = v.warehouse_id AND v3.retail_id = v.retail_id AND v3.date = v.date
LEFT JOIN
    product_rappi AS p ON p.sync_product = v.retail_id
LEFT JOIN
    wh_rappi AS w ON w.sync_wh = v.warehouse_id;"""
df_venta = pd.read_sql(venta, conn)
df_venta.head()

C:\Users\marcel.kempe\AppData\Local\Temp\ipykernel_29792\4098298884.py:102: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_venta = pd.read_sql(venta, conn)


,SYNC_WH,WAREHOUSEID,SYNC_PRODUCT,STOREREFERENCEID,VENTALW,VENTA_AYER,VENTA_2,VENTA_3
0,51,233,1058,427598,2.0,NaN,NaN,2.0
1,52,236,1058,427598,2.0,NaN,NaN,NaN
2,8,232,1058,427598,2.0,2.0,NaN,NaN
3,11,239,1058,427598,1.0,NaN,NaN,NaN
4,11,239,1058,427598,2.0,NaN,NaN,2.0


In [16]:
df_swa = pd.merge(left=df_swa,right=inventario,how='left',left_on=['SYNC_WH_ID','SYNC_PRODUCT_ID'],
                           right_on=['SYNC_WH','SYNC_ID'])
df_swa.head()


,CITY_NAME,WAREHOUSE_ID,SYNC_WH_ID,DEPENDENCIA,WAREHOUSE_NAME,STOREREFERENCE_ID,SYNC_PRODUCT_ID,EAN,IS_WL,SCOPE_WH,...,SWA_MISSING,RECEPTION_DATE,TRANSFORMATION_DATE,TRANSFOR_UNITS,SYNC_WH,WAREHOUSEID,STOREREFERENCEID,SYNC_ID,PRODUCT_NAME_y,INVENTARIO_TURBO
0,BogotC!Exito,180,12,4841,El Lago,431098,3396,2200000313515,1 Ideal,Bogotá Diamante,...,0.811165,2024-04-01,NaT,NaN,12,180,431098,3396,Banano EC x150 Gr - Experiencia Colombiana - ...,0.0
1,Medellín,236,52,4854,Tesoro,431087,3747,2200000313409,1 Ideal,MED TESORO,...,0.724086,2024-03-31,2024-03-31 11:08:10+00:00,711.0,52,236,431087,3747,Aguacate Hass Para Hoy EC x170 Gr - Experienc...,0.0
2,BogotC!Exito,181,9,4843,Parque 93,431098,3396,2200000313515,1 Ideal,TURBO FARMA BOGOTA,...,0.685761,2024-04-01,NaT,NaN,9,181,431098,3396,Banano EC x150 Gr - Experiencia Colombiana - ...,0.0
3,Barranquilla,244,63,4475,El Poblado Baq,431098,3396,2200000313515,1 Ideal,TURBOFARMA BQUILLA,...,0.523309,2024-04-01,NaT,NaN,63,244,431098,3396,Banano EC x150 Gr - Experiencia Colombiana - ...,204.0
4,Medellín,246,21,4474,Manila,431098,3396,2200000313515,1 Ideal,MED BIG MANILA,...,0.498626,2024-03-31,NaT,NaN,21,246,431098,3396,Banano EC x150 Gr - Experiencia Colombiana - ...,0.0


In [17]:
print(df_swa.columns)

Index(['CITY_NAME', 'WAREHOUSE_ID', 'SYNC_WH_ID', 'DEPENDENCIA',
       'WAREHOUSE_NAME', 'STOREREFERENCE_ID', 'SYNC_PRODUCT_ID', 'EAN',
       'IS_WL', 'SCOPE_WH', 'PRODUCT_NAME_x', 'MACROCATEGORY_NAME',
       'CATEGORY_NAME', 'SUBCATEGORY_NAME', 'SUPPLIER_NAME', 'TRADEMARK_NAME',
       'MAKER_NAME', 'BUCKET', 'SALES_AYER', 'SALES_28', 'AVL_NACIONAL',
       'SWA_NACIONAL', 'SWA_MISSING', 'RECEPTION_DATE', 'TRANSFORMATION_DATE',
       'TRANSFOR_UNITS', 'SYNC_WH', 'WAREHOUSEID', 'STOREREFERENCEID',
       'SYNC_ID', 'PRODUCT_NAME_y', 'INVENTARIO_TURBO'],
      dtype='object')


In [18]:
print("df_swa['WAREHOUSE_ID'] unique values:", df_swa['WAREHOUSE_ID'].unique())
print("df_ns1['WAREHOUSE_ID'] unique values:", df_ns1['WAREHOUSE_ID'].unique())


df_swa['WAREHOUSE_ID'] unique values: [180 236 181 244 246 257 232 260 254 243 233 263 273 295 182 248 297 262
 279 231 235 245 275 268 303 296 272 292 230 286 311 266 237 238 276 281
 264 271 333 239 277 285 298 328 282 309 265 259 283 334 331 332 329 330
 335]
df_ns1['WAREHOUSE_ID'] unique values: [ 93  47  24  12  63  17  33  48 107  14  18  19  50  79   9  52 104  66
  68  49   8  25  11   6 108  90   7  21   2  73  51  67  29  91 103  27
  70   5  23  55  28  72  60   3  26  84 106   4 105  78  16  58  61  31
 109]


In [19]:
print("Unique values in df_swa['SYNC_WH_ID']:")
print(df_swa['SYNC_WH_ID'].unique())

print("Unique values in df_swa['SYNC_PRODUCT_ID']:")
print(df_swa['SYNC_PRODUCT_ID'].unique())

print("Unique values in df_ns1['WAREHOUSE_ID']:")
print(df_ns1['WAREHOUSE_ID'].unique())

print("Unique values in df_ns1['STOREREFERENCEID']:")
print(df_ns1['STOREREFERENCEID'].unique())



Unique values in df_swa['SYNC_WH_ID']:
[ 12  52   9  63  21  48   8  29  68  49  51  24  31  79  47  67  93  78
   7  14  17  66  55  28  91   2   4   6  50  33   5  58  19  18  25  72
  70  16 107  11  27  61  90   3  60  84  26  23  73 108 105 106 103 104
 109]
Unique values in df_swa['SYNC_PRODUCT_ID']:
[ 3396  3747  4402  4155  4603  3965  5860  3404  8043  3972  6798 12769
  8599 12794  5392  8651  6633  4453  6115  6451 13758  2530  4111  4255
 12568  3950  4653  2036  5049 12707  3388  6765  5674  3554  6586 12406
 12940  6489  5980  8690  6721  4128  4049 12651  3545  6663  3792  4465
  4282 12893 13118 12221  9133 13658 12552  4118  2735  4586  7064 12882
  4256 11943  4512  4376 12958  6624  8709  4337  9050  3184  9141 13670
  8874  6290  8588  8773  9209  4206 13684  4623  4104  2711 12234  9277
  4638  6715 11574  6382  4200  1726  2702  8779  3727  9120  3661  6550
  9202  4407  8851 12349 13710 13018 14941 15013 12942  9990  4072 12589
  9061  8938 13633  4544  3996 1224

In [20]:
df_swa_1 = pd.merge(left=df_swa, right=df_ns1, how='left',
                    left_on=['SYNC_WH', 'SYNC_PRODUCT_ID'],
                    right_on=['WAREHOUSE_ID', 'STOREREFERENCEID'])

df_swa_1.head(10)

,CITY_NAME,WAREHOUSE_ID_x,SYNC_WH_ID,DEPENDENCIA,WAREHOUSE_NAME,STOREREFERENCE_ID,SYNC_PRODUCT_ID,EAN,IS_WL,SCOPE_WH,...,PLANNED_LW,DELIVERED_LW,PLANNED_AYER,DELIVERED_AYER,PLANNED_2,DELIVERED_2,PLANNED_3,DELIVERED_3,PLANNED_4,DELIVERED_4
0,BogotC!Exito,180,12,4841,El Lago,431098,3396,2200000313515,1 Ideal,Bogotá Diamante,...,2441.0,1457.0,110.0,110.0,NaN,NaN,858.0,287.0,NaN,NaN
1,Medellín,236,52,4854,Tesoro,431087,3747,2200000313409,1 Ideal,MED TESORO,...,1999.0,941.0,152.0,68.0,NaN,NaN,417.0,120.0,NaN,NaN
2,BogotC!Exito,181,9,4843,Parque 93,431098,3396,2200000313515,1 Ideal,TURBO FARMA BOGOTA,...,3033.0,2410.0,323.0,323.0,NaN,NaN,420.0,420.0,NaN,NaN
3,Barranquilla,244,63,4475,El Poblado Baq,431098,3396,2200000313515,1 Ideal,TURBOFARMA BQUILLA,...,2701.0,2013.0,531.0,531.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Medellín,246,21,4474,Manila,431098,3396,2200000313515,1 Ideal,MED BIG MANILA,...,1926.0,1384.0,319.0,215.0,NaN,NaN,240.0,163.0,NaN,NaN
5,Medellín,257,48,4438,La Frontera,431168,4402,2200000314253,1 Ideal,MED DIAMFARMA,...,1051.0,622.0,276.0,84.0,NaN,NaN,86.0,0.0,NaN,NaN
6,BogotC!Exito,232,8,4847,Country,431098,3396,2200000313515,1 Ideal,TURBO FARMA NIZA COUNTRY,...,1953.0,1621.0,307.0,307.0,NaN,NaN,61.0,61.0,NaN,NaN
7,Medellín,257,48,4438,La Frontera,431087,3747,2200000313409,1 Ideal,MED DIAMFARMA,...,1348.0,606.0,273.0,45.0,NaN,NaN,122.0,0.0,NaN,NaN
8,Medellín,260,29,4484,La Mota,431098,3396,2200000313515,1 Ideal,LA MOTA DIAMANTE,...,1308.0,824.0,349.0,0.0,NaN,NaN,138.0,138.0,NaN,NaN
9,Medellín,254,68,4487,Laureles,431087,3747,2200000313409,1 Ideal,MED DIAMFARMA,...,939.0,533.0,101.0,48.0,NaN,NaN,109.0,67.0,NaN,NaN


In [21]:
df_swa_1.columns

Index(['CITY_NAME', 'WAREHOUSE_ID_x', 'SYNC_WH_ID', 'DEPENDENCIA',
       'WAREHOUSE_NAME', 'STOREREFERENCE_ID', 'SYNC_PRODUCT_ID', 'EAN',
       'IS_WL', 'SCOPE_WH', 'PRODUCT_NAME_x', 'MACROCATEGORY_NAME',
       'CATEGORY_NAME', 'SUBCATEGORY_NAME', 'SUPPLIER_NAME', 'TRADEMARK_NAME',
       'MAKER_NAME', 'BUCKET', 'SALES_AYER', 'SALES_28', 'AVL_NACIONAL',
       'SWA_NACIONAL', 'SWA_MISSING', 'RECEPTION_DATE', 'TRANSFORMATION_DATE',
       'TRANSFOR_UNITS', 'SYNC_WH', 'WAREHOUSEID', 'STOREREFERENCEID_x',
       'SYNC_ID', 'PRODUCT_NAME_y', 'INVENTARIO_TURBO', 'WAREHOUSE_ID_y',
       'STOREREFERENCEID_y', 'PROVIDERID', 'PLANNED_LW', 'DELIVERED_LW',
       'PLANNED_AYER', 'DELIVERED_AYER', 'PLANNED_2', 'DELIVERED_2',
       'PLANNED_3', 'DELIVERED_3', 'PLANNED_4', 'DELIVERED_4'],
      dtype='object')

In [22]:
columnas_no_eliminar = ['CITY_NAME', 'WAREHOUSE_ID_x', 'SYNC_WH_ID', 'WAREHOUSE_NAME', 'STOREREFERENCE_ID', 'SYNC_PRODUCT_ID', 
                        'IS_WL', 'SCOPE_WH', 'PRODUCT_NAME_x','CATEGORY_NAME','SUPPLIER_NAME','MAKER_NAME','BUCKET', 'SALES_AYER', 'SALES_28', 'AVL_NACIONAL', 'SWA_NACIONAL',
       'SWA_MISSING',  'INVENTARIO_TURBO', 'PROVIDERID', 'PLANNED_LW', 'DELIVERED_LW', 'PLANNED_AYER', 'DELIVERED_AYER', 'PLANNED_2',
       'DELIVERED_2', 'PLANNED_3', 'DELIVERED_3', 'PLANNED_4', 'DELIVERED_4'  ]

df_swa_1.drop(columns=[col for col in df_swa_1 if col not in columnas_no_eliminar], inplace=True)

In [23]:
df_swa_1.columns

Index(['CITY_NAME', 'WAREHOUSE_ID_x', 'SYNC_WH_ID', 'WAREHOUSE_NAME',
       'STOREREFERENCE_ID', 'SYNC_PRODUCT_ID', 'IS_WL', 'SCOPE_WH',
       'PRODUCT_NAME_x', 'CATEGORY_NAME', 'SUPPLIER_NAME', 'MAKER_NAME',
       'BUCKET', 'SALES_AYER', 'SALES_28', 'AVL_NACIONAL', 'SWA_NACIONAL',
       'SWA_MISSING', 'INVENTARIO_TURBO', 'PROVIDERID', 'PLANNED_LW',
       'DELIVERED_LW', 'PLANNED_AYER', 'DELIVERED_AYER', 'PLANNED_2',
       'DELIVERED_2', 'PLANNED_3', 'DELIVERED_3', 'PLANNED_4', 'DELIVERED_4'],
      dtype='object')

In [24]:
len(df_swa_1)

4370

In [25]:
df_swa_2 = pd.merge(left=df_swa_1,right=df_merma_ayer,how='left',left_on=(['WAREHOUSE_ID_x','STOREREFERENCE_ID']),
                          right_on=('WAREHOUSEID','STOREREFERENCEID'))
df_swa_2.head()

,CITY_NAME,WAREHOUSE_ID_x,SYNC_WH_ID,WAREHOUSE_NAME,STOREREFERENCE_ID,SYNC_PRODUCT_ID,IS_WL,SCOPE_WH,PRODUCT_NAME_x,CATEGORY_NAME,...,PLANNED_3,DELIVERED_3,PLANNED_4,DELIVERED_4,WAREHOUSEID,STOREREFERENCEID,MERMA_LW,MERMA_AYER,MERMA_2,MERMA_3
0,BogotC!Exito,180,12,El Lago,431098,3396,1 Ideal,Bogotá Diamante,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,858.0,287.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Medellín,236,52,Tesoro,431087,3747,1 Ideal,MED TESORO,Aguacate Hass Para Hoy EC x170 Gr - Experienc...,Frutas y verduras,...,417.0,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BogotC!Exito,181,9,Parque 93,431098,3396,1 Ideal,TURBO FARMA BOGOTA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,420.0,420.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Barranquilla,244,63,El Poblado Baq,431098,3396,1 Ideal,TURBOFARMA BQUILLA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Medellín,246,21,Manila,431098,3396,1 Ideal,MED BIG MANILA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,240.0,163.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
len(df_swa_2)

4370

In [27]:
df_swa_2 = df_swa_2.drop_duplicates(subset=['WAREHOUSE_ID_x', 'STOREREFERENCE_ID'])


# Otra opción es especificar las columnas que quieres considerar para la detección de duplicados
# Por ejemplo, si quieres eliminar duplicados basándote en las columnas 'WAREHOUSE_ID' y 'STOREREFERENCE_ID'
# df_swa_2 = df_swa_2.drop_duplicates(subset=['WAREHOUSE_ID', 'STOREREFERENCE_ID'])

# Si quieres hacer la operación "inplace" (modificar el DataFrame original), puedes hacerlo así:
# df_swa_2.drop_duplicates(inplace=True)


In [28]:
len(df_swa_2)

4370

In [29]:
# Convierte las columnas a un tipo de datos común (por ejemplo, a cadena)
df_swa_2['SYNC_WH_ID'] = df_swa_2['SYNC_WH_ID'].astype(str)
df_swa_2['SYNC_PRODUCT_ID'] = df_swa_2['SYNC_PRODUCT_ID'].astype(str)
df_roq['WAREHOUSE_ID'] = df_roq['WAREHOUSE_ID'].astype(str)
df_roq['RETAIL_ID'] = df_roq['RETAIL_ID'].astype(str)

# Realiza la fusión
df_swa_4 = pd.merge(left=df_swa_2, right=df_roq, how='left',
                    left_on=['SYNC_WH_ID', 'SYNC_PRODUCT_ID'],
                    right_on=['WAREHOUSE_ID', 'RETAIL_ID'])
df_swa_4.head()


,CITY_NAME,WAREHOUSE_ID_x,SYNC_WH_ID,WAREHOUSE_NAME,STOREREFERENCE_ID,SYNC_PRODUCT_ID,IS_WL,SCOPE_WH,PRODUCT_NAME_x,CATEGORY_NAME,...,MOV_CHECK,ROQ_FINAL_BOXES,MOP,MOP_CHECK,AVAILABLE_TO_ORDER,MIN_ORDER_FILL,FILL_METHOD,FILLED_PERCENTAGE,ROQ_FINAL_UNITS_FILLED,ROQ_FINAL_VALUE_FILLED
0,BogotC!Exito,180,12,El Lago,431098,3396,1 Ideal,Bogotá Diamante,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,True,451.0,0.0,True,True,False,,0.0,451.0,261580.0
1,Medellín,236,52,Tesoro,431087,3747,1 Ideal,MED TESORO,Aguacate Hass Para Hoy EC x170 Gr - Experienc...,Frutas y verduras,...,True,185.0,0.0,True,True,False,,0.0,185.0,259000.0
2,BogotC!Exito,181,9,Parque 93,431098,3396,1 Ideal,TURBO FARMA BOGOTA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,True,581.0,0.0,True,True,False,,0.0,581.0,322455.0
3,Barranquilla,244,63,El Poblado Baq,431098,3396,1 Ideal,TURBOFARMA BQUILLA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,True,292.0,0.0,True,True,False,,0.0,292.0,160600.0
4,Medellín,246,21,Manila,431098,3396,1 Ideal,MED BIG MANILA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,True,246.0,0.0,True,True,False,,0.0,246.0,135300.0


In [30]:
print(df_swa_4.columns)

Index(['CITY_NAME', 'WAREHOUSE_ID_x', 'SYNC_WH_ID', 'WAREHOUSE_NAME',
       'STOREREFERENCE_ID', 'SYNC_PRODUCT_ID', 'IS_WL', 'SCOPE_WH',
       'PRODUCT_NAME_x', 'CATEGORY_NAME',
       ...
       'MOV_CHECK', 'ROQ_FINAL_BOXES', 'MOP', 'MOP_CHECK',
       'AVAILABLE_TO_ORDER', 'MIN_ORDER_FILL', 'FILL_METHOD',
       'FILLED_PERCENTAGE', 'ROQ_FINAL_UNITS_FILLED',
       'ROQ_FINAL_VALUE_FILLED'],
      dtype='object', length=131)


In [31]:
len(df_swa_4)

4370

In [32]:
df_swa_4 = df_swa_4.drop_duplicates(subset=['WAREHOUSE_ID_x', 'STOREREFERENCE_ID'])

# Otra opción es especificar las columnas que quieres considerar para la detección de duplicados
# Por ejemplo, si quieres eliminar duplicados basándote en las columnas 'WAREHOUSE_ID' y 'STOREREFERENCE_ID'
# df_swa_2 = df_swa_2.drop_duplicates(subset=['WAREHOUSE_ID', 'STOREREFERENCE_ID'])

# Si quieres hacer la operación "inplace" (modificar el DataFrame original), puedes hacerlo así:
# df_swa_2.drop_duplicates(inplace=True)


In [33]:
len(df_swa_4)

4370

In [34]:
# Convierte las columnas a un tipo de datos común (por ejemplo, a cadena)
df_swa_4['WAREHOUSE_ID_x'] = df_swa_4['WAREHOUSE_ID_x'].astype(str)
df_swa_4['STOREREFERENCE_ID'] = df_swa_4['STOREREFERENCE_ID'].astype(str)
df_forecast['WAREHOUSEID'] = df_forecast['WAREHOUSEID'].astype(str)
df_forecast['STOREREFERENCEID'] = df_forecast['STOREREFERENCEID'].astype(str)

# Realiza la fusión
df_swa_6 = pd.merge(left=df_swa_4, right=df_forecast, how='left',
                    left_on=['WAREHOUSE_ID_x', 'STOREREFERENCE_ID'],
                    right_on=['WAREHOUSEID', 'STOREREFERENCEID'])
df_swa_6.head()

,CITY_NAME,WAREHOUSE_ID_x,SYNC_WH_ID,WAREHOUSE_NAME,STOREREFERENCE_ID,SYNC_PRODUCT_ID,IS_WL,SCOPE_WH,PRODUCT_NAME_x,CATEGORY_NAME,...,WAREHOUSEID_y,STOREREFERENCEID_y,FORECAST_LW,SALES_LW,FORECAST_AYER,SALES_AYER_y,FORECAST_MENOS2,SALES_MENOS2,FORECAST_MENOS3,SALES_MENOS3
0,BogotC!Exito,180,12,El Lago,431098,3396,1 Ideal,Bogotá Diamante,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,180,431098,1191.0,1011.0,313.0,0.0,214.0,48.0,179.0,78.0
1,Medellín,236,52,Tesoro,431087,3747,1 Ideal,MED TESORO,Aguacate Hass Para Hoy EC x170 Gr - Experienc...,Frutas y verduras,...,236,431087,1152.0,762.0,142.0,0.0,197.0,67.0,94.0,124.0
2,BogotC!Exito,181,9,Parque 93,431098,3396,1 Ideal,TURBO FARMA BOGOTA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,181,431098,1854.0,1801.0,395.0,31.0,288.0,196.0,271.0,400.0
3,Barranquilla,244,63,El Poblado Baq,431098,3396,1 Ideal,TURBOFARMA BQUILLA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,244,431098,1566.0,1032.0,187.0,0.0,248.0,0.0,182.0,0.0
4,Medellín,246,21,Manila,431098,3396,1 Ideal,MED BIG MANILA,Banano EC x150 Gr - Experiencia Colombiana - ...,Frutas y verduras,...,246,431098,1167.0,1033.0,170.0,3.0,182.0,211.0,157.0,137.0


In [35]:
len(df_swa_6)

4370

In [36]:
df_swa_6.to_excel("fruver_diario.xlsx", index=False)


In [37]:
import smtplib, getpass, os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from datetime import datetime
from email import encoders

user = "marcel.kempe@rappi.com"  # Reemplaza con tu dirección de correo
password = "lhhd ievh mjbu isye"  # Reemplaza con tu contraseña

fecha = datetime.today().strftime('%Y-%m-%d')

#Ajustes del mail
remitente1 = """Rappi Turbo Supply"""

#destinatario1 = ['marcel.kempe@rappi.com', 'mariangela.uribe@rappi.com', 'juliana.zarate@rappi.com', 'valeria.leguizamon@lum.com.co', 'johan.mayorga@rappi.com']
destinatario1 = ['marcel.kempe@rappi.com']
asunto1 = f"""Informe fruver {fecha}"""
mensaje1 = f"""Hola, <br><br> Adjunto información de la categorías al día {fecha}.<br><br> ¡Gracias!"""

#Host y puerto SMTP de Gmail
server1 = smtplib.SMTP('smtp.gmail.com', 587)

#protocolo de cifrado de datos
server1.starttls()

#Credenciales
server1.login(user, password)

#Muestra la depuración de la operación de envío 1=true
server1.set_debuglevel(1)

msg1 = MIMEMultipart()
msg1['Subject'] = asunto1
msg1['From'] = remitente1
msg1['To'] = ", ".join(destinatario1)


mensaje1 = MIMEText(mensaje1, 'html')
msg1.attach(mensaje1)

part1 = MIMEBase('application', "octet-stream")
part1.set_payload(open("fruver_diario.xlsx", "rb").read())
encoders.encode_base64(part1)

part1.add_header('Content-Disposition', 'attachment; filename="fruver_diario.xlsx"')
msg1.attach(part1)

#Enviar mail
server1.sendmail(remitente1, destinatario1, msg1.as_string())

#Cerrar la conexión
server1.quit()

send: 'mail FROM:<Rappi> size=2694811\r\n'
reply: b'250 2.1.0 OK r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp\r\n'
reply: retcode (250); Msg: b'2.1.0 OK r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp'
send: 'rcpt TO:<marcel.kempe@rappi.com>\r\n'
reply: b'250 2.1.5 OK r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp\r\n'
reply: retcode (250); Msg: b'2.1.5 OK r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp'
send: 'data\r\n'
reply: b'354  Go ahead r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp\r\n'
reply: retcode (354); Msg: b'Go ahead r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp'
data: (354, b'Go ahead r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp')
send: b'Content-Type: multipart/mixed; boundary="===============1140066107217572704=="\r\nMIME-Version: 1.0\r\nSubject: Informe fruver 2024-04-02\r\nFrom: Rappi Turbo Supply\r\nTo: marcel.kempe@rappi.com\r\n\r\n--===============1140066107217572704==\r\nConte

(221,
 b'2.0.0 closing connection r9-20020a814409000000b0061511f88a5fsm579474ywa.141 - gsmtp')

# conn.close()